# Assignment 01: Dataset Visualization and Descriptive Analysis

This notebook implements the Data Mining Assignment 01 tasks using the Kaggle Diabetes dataset. The workflow covers dataset loading, feature selection, NumPy conversion, descriptive statistics, and interactive visualization using Plotly.

**Dataset:** Diabetes Dataset from Kaggle (`mathchi/diabetes-data-set`)  
**Target label:** `Outcome`  
**Primary selected dimensions:** first three real-valued features plus the class label


## 1. Imports and Project Paths

This section imports the required libraries and defines output folders. HTML versions of the interactive Plotly figures are saved under `outputs/html/`.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

try:
    import kagglehub
except ImportError:
    kagglehub = None

# Resolve project root whether the notebook is run from the root folder or from notebooks/.
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR
OUTPUT_DIR = PROJECT_ROOT / "outputs"
HTML_DIR = OUTPUT_DIR / "html"
DATA_DIR = PROJECT_ROOT / "data"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
HTML_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

DATASET_ID = "mathchi/diabetes-data-set"
DATA_FILE = "diabetes.csv"


## 2. Load the Dataset

Assignment tasks 1 and 2 require downloading the assigned dataset and reading it into a pandas DataFrame. The code first checks common local paths. If the file is not available locally, it downloads the dataset using `kagglehub`.


In [ ]:
def locate_dataset(filename: str = DATA_FILE) -> Path:
    """Find the dataset locally or download it with kagglehub."""
    candidate_paths = [
        DATA_DIR / filename,
        PROJECT_ROOT / filename,
        CURRENT_DIR / filename,
        Path("/content") / filename,
        Path("/kaggle/input/diabetes-data-set") / filename,
    ]

    for candidate in candidate_paths:
        if candidate.exists():
            return candidate

    if kagglehub is None:
        raise ImportError(
            "kagglehub is not installed and no local diabetes.csv file was found. "
            "Install kagglehub or place diabetes.csv in the data/ folder."
        )

    downloaded_path = Path(kagglehub.dataset_download(DATASET_ID))
    candidate = downloaded_path / filename
    if not candidate.exists():
        raise FileNotFoundError(f"Could not find {filename} inside {downloaded_path}")
    return candidate


dataset_path = locate_dataset()
df = pd.read_csv(dataset_path)

print("Dataset path:", dataset_path)
print("Shape:", df.shape)
df.head()


## 3. Basic Data Quality Check

This section checks for missing values and negative values before selecting the working features.


In [ ]:
missing_counts = df.isnull().sum()
negative_counts = (df.select_dtypes(include=[np.number]) < 0).sum()

print("Missing values per column:")
print(missing_counts)

print("
Negative values per numeric column:")
print(negative_counts)


## 4. Remove Identifier Columns and Select Required Dimensions

Assignment tasks 3 and 4 ask us to remove identifier columns and keep the first three real-valued dimensions plus the class label. In this dataset, there is no explicit ID/name/serial column, so the first three numeric features are selected directly along with `Outcome`.


In [ ]:
def is_identifier_column(column_name: str) -> bool:
    """Return True if a column name looks like an identifier rather than a feature."""
    keywords = ["id", "name", "identifier", "serial", "index", "patient", "record"]
    normalized = str(column_name).lower()
    return any(keyword in normalized for keyword in keywords)


label_col = df.columns[-1]
identifier_cols = [col for col in df.columns[:-1] if is_identifier_column(col)]

numeric_feature_cols = [
    col for col in df.columns[:-1]
    if pd.api.types.is_numeric_dtype(df[col]) and col not in identifier_cols
]

selected_feature_cols = numeric_feature_cols[:3]
selected_cols = selected_feature_cols + [label_col]
working_df = df[selected_cols].copy()

working_csv_path = OUTPUT_DIR / "diabetes_first_three_dimensions.csv"
working_df.to_csv(working_csv_path, index=False)

print("Identifier columns removed:", identifier_cols if identifier_cols else "None found")
print("Selected feature columns:", selected_feature_cols)
print("Class label column:", label_col)
print("Saved working dataset to:", working_csv_path)
working_df.head()


## 5. Convert the DataFrame to a NumPy Array

Assignment task 5 requires converting the selected DataFrame into a NumPy array. The class labels are already numeric in the Diabetes dataset: `0` for non-diabetic and `1` for diabetic.


In [ ]:
class_values = sorted(working_df[label_col].unique())
class_names = ["Not diabetic" if value == 0 else "Diabetic" if value == 1 else str(value) for value in class_values]
class_name_lookup = dict(zip(class_values, class_names))

data_array = working_df.to_numpy()

print("NumPy array shape:", data_array.shape)
print("Class labels:", class_name_lookup)
print("First five rows:")
print(data_array[:5])


## 6. Plotting Helper

All plots are generated with Plotly. The helper saves each interactive figure as an HTML file and displays it inside the notebook.


In [ ]:
def save_and_show(fig, filename: str):
    """Save an interactive Plotly figure as HTML and display it."""
    html_path = HTML_DIR / filename
    fig.write_html(html_path, include_plotlyjs="cdn")
    print("Saved:", html_path)
    fig.show()


plot_df = working_df.copy()
plot_df["Class"] = plot_df[label_col].map(class_name_lookup)

feature_cols_2d = selected_feature_cols[:2]
feature_cols_3d = selected_feature_cols[:3]

symbol_sequence_2d = ["circle", "square", "triangle-up", "diamond", "cross", "x"]
symbol_map_2d = {
    class_name_lookup[value]: symbol_sequence_2d[index % len(symbol_sequence_2d)]
    for index, value in enumerate(class_values)
}

symbol_sequence_3d = ["circle", "square", "diamond", "cross", "x"]
symbol_map_3d = {
    class_name_lookup[value]: symbol_sequence_3d[index % len(symbol_sequence_3d)]
    for index, value in enumerate(class_values)
}


## 7. Interactive 2D Scatter Plot

Assignment task 7 uses the first two dimensions and class-specific marker shapes.


In [ ]:
fig = px.scatter(
    plot_df,
    x=feature_cols_2d[0],
    y=feature_cols_2d[1],
    color="Class",
    symbol="Class",
    symbol_map=symbol_map_2d,
    title=f"2D Scatter Plot: {feature_cols_2d[0]} vs {feature_cols_2d[1]}",
)
fig.update_traces(marker=dict(size=7, opacity=0.85))
save_and_show(fig, "q07_interactive_2d_scatter.html")


## 8. Interactive 3D Scatter Plot

Assignment task 8 uses the first three dimensions and class-specific marker shapes.


In [ ]:
fig = px.scatter_3d(
    plot_df,
    x=feature_cols_3d[0],
    y=feature_cols_3d[1],
    z=feature_cols_3d[2],
    color="Class",
    symbol="Class",
    symbol_map=symbol_map_3d,
    title=f"3D Scatter Plot: {feature_cols_3d[0]}, {feature_cols_3d[1]}, {feature_cols_3d[2]}",
)
fig.update_traces(marker=dict(size=5, opacity=0.85))
save_and_show(fig, "q08_interactive_3d_scatter.html")


## 9. 2D Scatter Plot with Class Mean Points

Assignment task 9 repeats the 2D plot and overlays the mean data point for each class using the same marker shape and a larger marker size.


In [ ]:
means_2d = plot_df.groupby(label_col)[feature_cols_2d].mean().reset_index()
means_2d["Class"] = means_2d[label_col].map(class_name_lookup)

fig = px.scatter(
    plot_df,
    x=feature_cols_2d[0],
    y=feature_cols_2d[1],
    color="Class",
    symbol="Class",
    symbol_map=symbol_map_2d,
    title=f"2D Scatter Plot with Class Means: {feature_cols_2d[0]} vs {feature_cols_2d[1]}",
)
fig.update_traces(marker=dict(size=7, opacity=0.85))

for _, row in means_2d.iterrows():
    fig.add_trace(go.Scatter(
        x=[row[feature_cols_2d[0]]],
        y=[row[feature_cols_2d[1]]],
        mode="markers",
        name=f"{row['Class']} mean",
        marker=dict(
            size=18,
            symbol=symbol_map_2d[row["Class"]],
            line=dict(width=2, color="black"),
        ),
    ))

save_and_show(fig, "q09_2d_scatter_with_class_means.html")
means_2d


## 10. 3D Scatter Plot with Class Mean Points

Assignment task 10 repeats the class-mean overlay for the first three dimensions.


In [ ]:
means_3d = plot_df.groupby(label_col)[feature_cols_3d].mean().reset_index()
means_3d["Class"] = means_3d[label_col].map(class_name_lookup)

fig = px.scatter_3d(
    plot_df,
    x=feature_cols_3d[0],
    y=feature_cols_3d[1],
    z=feature_cols_3d[2],
    color="Class",
    symbol="Class",
    symbol_map=symbol_map_3d,
    title=f"3D Scatter Plot with Class Means: {feature_cols_3d[0]}, {feature_cols_3d[1]}, {feature_cols_3d[2]}",
)
fig.update_traces(marker=dict(size=5, opacity=0.85))

for _, row in means_3d.iterrows():
    fig.add_trace(go.Scatter3d(
        x=[row[feature_cols_3d[0]]],
        y=[row[feature_cols_3d[1]]],
        z=[row[feature_cols_3d[2]]],
        mode="markers",
        name=f"{row['Class']} mean",
        marker=dict(
            size=14,
            symbol=symbol_map_3d[row["Class"]],
            line=dict(width=2, color="black"),
        ),
    ))

save_and_show(fig, "q10_3d_scatter_with_class_means.html")
means_3d


## 11. Histograms for the First Three Dimensions

Assignment task 11 visualizes the distribution of each selected dimension.


In [ ]:
for feature in feature_cols_3d:
    fig = px.histogram(
        plot_df,
        x=feature,
        color="Class",
        barmode="overlay",
        opacity=0.7,
        title=f"Histogram of {feature} by Class",
    )
    fig.update_layout(bargap=0.05, xaxis_title=feature, yaxis_title="Count")
    save_and_show(fig, f"q11_histogram_{feature.lower().replace(' ', '_')}.html")


## 12. Standard Deviation for the First Three Dimensions

Assignment task 12 calculates sigma for the selected dimensions.


In [ ]:
std_values = plot_df[feature_cols_3d].std()
print("Standard deviation for the first three selected features:")
print(std_values.round(4))


## 13. Boxplots for the First Three Dimensions

Assignment task 13 draws boxplots for all data points across the first three dimensions.


In [ ]:
fig = px.box(
    plot_df,
    y=feature_cols_3d,
    points="outliers",
    title="Boxplots for the First Three Features",
    labels={"variable": "Feature", "value": "Value"},
)
fig.update_layout(xaxis_title="Features", yaxis_title="Values", showlegend=False)
save_and_show(fig, "q13_boxplots_first_three_features.html")


## 14. Boxplots Grouped by Class

Assignment task 14 compares the same three dimensions separately for each class.


In [ ]:
melted_df = plot_df.melt(
    id_vars=["Class"],
    value_vars=feature_cols_3d,
    var_name="Feature",
    value_name="Value",
)

fig = px.box(
    melted_df,
    x="Feature",
    y="Value",
    color="Class",
    points="outliers",
    title="Boxplots by Class for the First Three Features",
)
fig.update_layout(boxmode="group", xaxis_title="Features", yaxis_title="Values")
save_and_show(fig, "q14_boxplots_by_class.html")


## 15. Scatterplot Matrix

Assignment task 15 requires a 3 × 3 matrix plot for the first three dimensions.


In [ ]:
fig = px.scatter_matrix(
    plot_df,
    dimensions=feature_cols_3d,
    color="Class",
    title="Scatterplot Matrix for the First Three Features",
)
fig.update_traces(diagonal_visible=True, marker=dict(size=4, opacity=0.8))
fig.update_layout(height=700, width=900)
save_and_show(fig, "q15_scatter_matrix.html")


## 16. Circle Segment Plot for the First Seven Real-Valued Dimensions

Assignment task 16 uses the first seven real-valued dimensions. Values are normalized to a 0-1 scale so features with different units can be compared on the same circular plot.


In [ ]:
full_numeric_features = [
    col for col in df.columns
    if col != label_col and pd.api.types.is_numeric_dtype(df[col]) and not is_identifier_column(col)
]
first7_features = full_numeric_features[:7]

normalized_df = df[first7_features].copy()
normalized_df = (normalized_df - normalized_df.min()) / (normalized_df.max() - normalized_df.min() + 1e-12)
theta = np.linspace(0, 360, len(first7_features), endpoint=False)

max_points = 300
sample_indices = np.arange(len(normalized_df))
if len(sample_indices) > max_points:
    rng = np.random.default_rng(0)
    sample_indices = rng.choice(sample_indices, size=max_points, replace=False)

normalized_sample = normalized_df.iloc[sample_indices].reset_index(drop=True)

fig = go.Figure()
for row_index in range(len(normalized_sample)):
    radius_values = normalized_sample.iloc[row_index].to_numpy()
    fig.add_trace(go.Scatterpolar(
        r=np.r_[radius_values, radius_values[0]],
        theta=np.r_[theta, theta[0]],
        mode="lines",
        line=dict(width=1),
        opacity=0.08,
        showlegend=False,
    ))

for class_value in class_values:
    class_mean = normalized_df[df[label_col] == class_value].mean().to_numpy()
    fig.add_trace(go.Scatterpolar(
        r=np.r_[class_mean, class_mean[0]],
        theta=np.r_[theta, theta[0]],
        mode="lines+markers",
        line=dict(width=3),
        marker=dict(size=5),
        name=f"{class_name_lookup[class_value]} mean",
    ))

fig.update_layout(
    title=f"Circle Segment Plot for the First {len(first7_features)} Real-Valued Features",
    polar=dict(
        radialaxis=dict(visible=True, range=[0, 1]),
        angularaxis=dict(tickmode="array", tickvals=theta, ticktext=first7_features),
    ),
    showlegend=True,
)
save_and_show(fig, "q16_circle_segment_plot.html")


## 17. Parallel Coordinates Plot

Assignment task 17 draws seven parallel axes for the first seven dimensions. Lines are colored by the class label.


In [ ]:
dimensions = []
for feature in first7_features:
    dimensions.append(dict(
        range=[float(df[feature].min()), float(df[feature].max())],
        label=feature,
        values=df[feature],
    ))

fig = go.Figure(data=go.Parcoords(
    line=dict(
        color=df[label_col],
        colorscale=[[0, "#1f77b4"], [1, "#d62728"]],
        cmin=df[label_col].min(),
        cmax=df[label_col].max(),
        showscale=True,
        colorbar=dict(title="Class"),
    ),
    dimensions=dimensions,
))
fig.update_layout(title=f"Parallel Coordinates Plot for the First {len(first7_features)} Features")
save_and_show(fig, "q17_parallel_coordinates.html")

summary_stats = pd.DataFrame({
    "mean": df[first7_features].mean(),
    "std": df[first7_features].std(),
})
summary_stats.round(4)
